# TTA Example

> Usage

```bash
uv run jupyter nbconvert --to script example.ipynb
uv run example.py --dataset shift --model rcnn --method norm_engine --device 0
```

## Imports and Configs

In [ ]:
import sys
from argparse import ArgumentParser
from contextlib import redirect_stdout, nullcontext
from os import path, environ, system, makedirs, devnull

import torch
from torchinfo import summary

from ttadapters import datasets
from ttadapters import models, methods
from ttadapters.datasets import scenarios
from ttadapters.utils.validator import DetectionEvaluator
from ttadapters.utils.visualizer import visualize_metrics

In [ ]:
import pandas as pd

pd.options.display.float_format = lambda x: f"{x*100 if x < 1 else x:.4f}"

In [ ]:
environ["TORCHDYNAMO_CAPTURE_SCALAR_OUTPUTS"] = "1"
environ["TORCHDYNAMO_CAPTURE_DYNAMIC_OUTPUT_SHAPE_OPS"] = "1"

torch._dynamo.config.capture_scalar_outputs = True
torch._dynamo.config.suppress_errors = True

### Check GPU Availability

In [ ]:
_ = system("nvidia-smi")

### Parse Arguments

In [ ]:
ABLATION_NUM = 3

In [ ]:
# Set Block Count
N_BLOCKS = 1

In [ ]:
# Set Batch Size
BATCH_SIZE = 1  # Online
FIT_BATCH_SIZE = 40

# Set CUDA Device Number
DEVICE_NUM = 0

# Set Total Rounds
TOTAL_ROUNDS = 1

# Set Data Root
DATA_ROOT = path.join(".", "data")
RESULT_ROOT = path.join(".", "results", "ablations", f"ab{ABLATION_NUM}")

# Set Target Dataset
SOURCE_DOMAIN = datasets.SHIFTDataset
DISABLE_DATALOG = False

# Set Model List
MODEL_ZOO = ["rcnn", "swinrcnn"]
MODEL_TYPE = MODEL_ZOO[0]

# Set method
METHOD_ZOO = ["pit_engine"]
METHOD_TYPE = METHOD_ZOO[0]

In [ ]:
# Create argument parser
parser = ArgumentParser(description="Adaptation experiment script for Test-Time Adapters")

# Add model arguments
parser.add_argument("--dataset", type=str, choices=["shift", "city"], default="shift", help="Target dataset")
parser.add_argument("--model", type=str, choices=MODEL_ZOO, default=MODEL_TYPE, help="Model architecture")
parser.add_argument("--method", type=str, choices=METHOD_ZOO, default=METHOD_TYPE, help="Method")
parser.add_argument("--blocks", type=int, choices=[1, 2, 3, 4, 5], default=N_BLOCKS, help="Number of blocks")

# Add training arguments
parser.add_argument("--adapt-batch", type=int, default=BATCH_SIZE, help="Adaptation batch size")
parser.add_argument("--fit-batch", type=int, default=FIT_BATCH_SIZE, help="Engine fitting batch size")
parser.add_argument("--total-rounds", type=int, default=TOTAL_ROUNDS, help="Total rounds")
parser.add_argument("--data-root", type=str, default=DATA_ROOT, help="Root directory for datasets")
parser.add_argument("--results-root", type=str, default=RESULT_ROOT, help="Root directory for adaptation results")
parser.add_argument("--device", type=int, default=0, help="CUDA device number")
parser.add_argument("--disable-datalog", action="store_true", help="Disable datalog")

# Parsing arguments
if "ipykernel" in sys.modules:
    args = parser.parse_args([])
    print("INFO: Running in notebook mode with default arguments")
else:
    args = parser.parse_args()

# Configure device
DEVICE_NUM = 0 if not args.device else args.device
environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE_NUM)
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
    DEVICE_NUM = -1
print(f"INFO: Using device - {device}:{DEVICE_NUM}")

# Update global variables based on parsed arguments
BATCH_SIZE = args.adapt_batch
FIT_BATCH_SIZE = args.fit_batch
TOTAL_ROUNDS = args.total_rounds
DATA_ROOT = args.data_root
DISABLE_DATALOG = args.disable_datalog
DISABLE_DATALOG_CTX = redirect_stdout(open(devnull, "w")) if DISABLE_DATALOG else nullcontext()
RESULT_ROOT = args.results_root
MODEL_TYPE = args.model
METHOD_TYPE = args.method
match args.dataset:
    case "shift":
        SOURCE_DOMAIN = datasets.SHIFTDataset
        if DISABLE_DATALOG:
            import logging
            logging.getLogger("shift_dev_logger").setLevel(logging.CRITICAL)
    case "city":
        SOURCE_DOMAIN = datasets.CityScapesDataset
    case _:
        raise ValueError(f"Unsupported dataset: {args.dataset}")

print(f"INFO: Running online adaptation with batch size {BATCH_SIZE}")

N_BLOCKS = args.blocks
print(f"INFO: Using {N_BLOCKS} blocks")

## Define Dataset

In [ ]:
# Fast download patch
datasets.patch_fast_download_for_object_detection()

In [ ]:
# Ensure split (required due to Scenario class works with coroutines)
with DISABLE_DATALOG_CTX:
    match SOURCE_DOMAIN:
        case datasets.SHIFTDataset:
            train_dataset = datasets.SHIFTContinuousSubsetForObjectDetection(root=DATA_ROOT, train=True)
        case datasets.CityScapesDataset:
            train_dataset = datasets.CityScapesDatasetForObjectDetection(root=DATA_ROOT, train=True)
        case _:
            raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

# Dataset info
CLASSES = train_dataset.classes
NUM_CLASSES = len(CLASSES)
print(f"INFO: Number of classes - {NUM_CLASSES} {CLASSES}")

## Load Base Model

In [ ]:
# Initialize base_model
DATA_TYPE = torch.float32
match MODEL_TYPE:
    case "rcnn":
        base_model = models.FasterRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case "swinrcnn":
        base_model = models.SwinRCNNForObjectDetection(dataset=SOURCE_DOMAIN)
        load_result = base_model.load_from(**vars(base_model.Weights.SHIFT_CLEAR_NATUREYOO if SOURCE_DOMAIN == datasets.SHIFTDataset else base_model.Weights.CITYSCAPES), strict=False)
    case _:
        raise ValueError(f"Unsupported model type: {MODEL_TYPE}")

data_preparation = base_model.DataPreparation(train_dataset, evaluation_mode=True)
print(f"INFO: Using data precision - {DATA_TYPE}")
print("INFO: Model state loaded -", load_result)
base_model.to(device)

In [ ]:
summary(base_model)

## Load Adaptation Method

In [ ]:
from ttadapters.methods.cascaded import PITEngine

In [ ]:
from enum import Enum


class TargetKeyPreset(Enum):
    """
    Preset patterns for cascade_target.
    Strategies are applied to the **EARLY Blocks** of the backbone to fast optimization.

    Block structure reference
    ─────────────────────────
    ResNet BottleneckBlock (Detectron2 / RT-DETR):
        res2.0  →  conv1.norm / conv2.norm / conv3.norm   (block 0)
        res2.1  →  conv1.norm / conv2.norm / conv3.norm   (block 1)
        res2.2  →  conv1.norm / conv2.norm / conv3.norm   (block 2)
        res3.0  →  conv1.norm / conv2.norm / conv3.norm   (block 3)
        res3.1  →  conv1.norm / conv2.norm / conv3.norm   (block 4)
        * "1 block" = 3 conv norms in the same residual stage

    Swin SwinTransformerBlock:
        layer0.blocks.0  →  norm1, norm2   (block 0)
        layer0.blocks.1  →  norm1, norm2   (block 1)
        layer1.blocks.0  →  norm1, norm2   (block 2)
        layer1.blocks.1  →  norm1, norm2   (block 3)
        layer2.blocks.0  →  norm1, norm2   (block 4)
        * "1 block" = 2 norms (norm1 + norm2)
    """

    # ── Default presets (original) ──────────────────────────────────────────

    RESNET = [  # res2 (3 blocks) / stages 0 = 3 blocks
        r"\.res2.*\.conv[123]\.norm$",               # BottleneckBlock (Detectron2)
        r"\.stages\.0.*\.layer\.[012]\.normalization$",  # RTDetrResNetBottleNeckLayer (RT-DETR)
    ]

    SWIN = [  # layer0 (2 blocks) + layer1 (2 blocks) = 4 blocks
        r"\.layers\.[01]\.blocks\..*\.norm[12]$",    # SwinTransformerBlock
    ]

    # ── Ablation study presets: B1 ~ B5 ─────────────────────────────────────

    RESNET_B1 = [  # 1 block  →  res2.0 only
        r"\.res2\.0\.conv[123]\.norm$",
        r"\.stages\.0\.layers\.0\.layer\.[012]\.normalization$",
    ]

    RESNET_B2 = [  # 2 blocks →  res2.0 ~ res2.1
        r"\.res2\.[01]\.conv[123]\.norm$",
        r"\.stages\.0\.layers\.[01]\.layer\.[012]\.normalization$",
    ]

    RESNET_B3 = [  # 3 blocks →  res2.* (identical to RESNET)
        r"\.res2.*\.conv[123]\.norm$",
        r"\.stages\.0.*\.layer\.[012]\.normalization$",
    ]

    RESNET_B4 = [  # 4 blocks →  res2.* + res3.0
        r"\.res2.*\.conv[123]\.norm$",
        r"\.res3\.0\.conv[123]\.norm$",
        r"\.stages\.0.*\.layer\.[012]\.normalization$",
        r"\.stages\.1\.layers\.0\.layer\.[012]\.normalization$",
    ]

    RESNET_B5 = [  # 5 blocks →  res2.* + res3.0 ~ res3.1
        r"\.res2.*\.conv[123]\.norm$",
        r"\.res3\.[01]\.conv[123]\.norm$",
        r"\.stages\.0.*\.layer\.[012]\.normalization$",
        r"\.stages\.1\.layers\.[01]\.layer\.[012]\.normalization$",
    ]

    # ── Swin ablation ────────────────────────────────────────────────────────

    SWIN_B1 = [  # 1 block  →  layer0.blocks.0
        r"\.layers\.0\.blocks\.0\.norm[12]$",
    ]

    SWIN_B2 = [  # 2 blocks →  layer0.blocks.0~1
        r"\.layers\.0\.blocks\.[01]\.norm[12]$",
    ]

    SWIN_B3 = [  # 3 blocks →  layer0.blocks.0~1 + layer1.blocks.0
        r"\.layers\.0\.blocks\.[01]\.norm[12]$",
        r"\.layers\.1\.blocks\.0\.norm[12]$",
    ]

    SWIN_B4 = [  # 4 blocks →  layer0 + layer1 (identical to SWIN)
        r"\.layers\.[01]\.blocks\..*\.norm[12]$",
    ]

    SWIN_B5 = [  # 5 blocks →  layer0 + layer1 + layer2.blocks.0
        r"\.layers\.[01]\.blocks\..*\.norm[12]$",
        r"\.layers\.2\.blocks\.0\.norm[12]$",
    ]


# Convenience lookup: model_type × n_blocks → preset
ABLATION_PRESETS: dict[tuple[str, int], TargetKeyPreset] = {
    ("rcnn",     1): TargetKeyPreset.RESNET_B1,
    ("rcnn",     2): TargetKeyPreset.RESNET_B2,
    ("rcnn",     3): TargetKeyPreset.RESNET_B3,
    ("rcnn",     4): TargetKeyPreset.RESNET_B4,
    ("rcnn",     5): TargetKeyPreset.RESNET_B5,
    ("swinrcnn", 1): TargetKeyPreset.SWIN_B1,
    ("swinrcnn", 2): TargetKeyPreset.SWIN_B2,
    ("swinrcnn", 3): TargetKeyPreset.SWIN_B3,
    ("swinrcnn", 4): TargetKeyPreset.SWIN_B4,
    ("swinrcnn", 5): TargetKeyPreset.SWIN_B5,
}

In [ ]:
from typing import Literal
from dataclasses import dataclass, field

from ttadapters.methods.base import AdaptationConfig


@dataclass
class PITConfig(AdaptationConfig):
    """Configuration for PITConfig."""
    adaptation_name = "PITEngine"

    adapt_lr: float = 1e-3
    optim: Literal["SGD", "Adam", "AdamW"] = "Adam"

    # Engine configuration
    itm_type: Literal["clahe", "gamma", "clahe-gamma", "clahe-gamma-residual"] = "gamma"
    cascade_target: list[str] = None
    exclude_target: list[str] = field(default_factory=lambda: ["stem", "patch_embed", "embedder"])
    disable_blending: bool = False
    blend_ratio: float = 0.6
    mask_value: int = 114  # YOLO11 default padding value
    masked_processing: bool = False

    # CLAHE parameters
    clahe_clip_limit: float = 2.0
    clahe_tile_size: int = 8

    # Gamma parameters
    use_differentiable_stretch: bool = True
    gamma_temperature: float = 0.01
    gamma_range: tuple[float, float] = (0.5, 2.0)  # *2 to /2
    gamma_noise_floor: float = 0.0
    gamma_saturation_limit: float = 100.0

    # Anchor configuration
    use_kl_divergence: bool = True  # if false, use MSE loss

In [ ]:
config = PITConfig(cascade_target=ABLATION_PRESETS[MODEL_TYPE, N_BLOCKS].value)

In [ ]:
adaptive_model = PITEngine(config, base_model=base_model)
adaptive_model.to(device)

### Fit engine with source if required

In [ ]:
adaptive_model.fit(data_preparation, batch_size=FIT_BATCH_SIZE, shuffle=False)

## Evaluation

In [ ]:
# Load Pretrained APT Weights & Un-Freeze Model Encoder
# Allow FPN/Encoder to adapt during online adaptation
base_model.eval()
adaptive_model.online()
summary(adaptive_model)

In [ ]:
def get_save_path(scenario, round=None):
    suffix = f"_r{round}" if round else ""
    blocks = f"_blk{N_BLOCKS}_anchr{len(adaptive_model.dist_norm.anchors)}"
    prefix = path.join(RESULT_ROOT, MODEL_TYPE, scenario.__class__.__name__.lower(), adaptive_model.model_type)
    makedirs(prefix, exist_ok=True)
    return path.join(prefix, "model" + blocks + suffix + ".pkl"), path.join(prefix, "result" + blocks + suffix + ".json")

In [ ]:
import json

def make_json_serializable(result):
    if isinstance(result, list):
        return [make_json_serializable(item) for item in result]
    elif isinstance(result, dict):
        return {(k.value if hasattr(k, 'value') else k): make_json_serializable(v) for k, v in result.items()}
    return result

def save_result_json(result, result_path):
    with open(result_path, "w", encoding="utf-8") as f:
        json.dump(make_json_serializable(result), f)

### Load Scenarios

In [ ]:
with DISABLE_DATALOG_CTX:
    match SOURCE_DOMAIN:
        case datasets.SHIFTDataset:
            continual_scenario = scenarios.SHIFTDiscreteScenarioForContinualTTA(
                root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None
            )
        case datasets.CityScapesDataset:
            continual_scenario = scenarios.CityScapesDiscreteScenarioForContinualTTA(
                root=DATA_ROOT, valid=True, transforms=data_preparation.transforms, order=None
            )
        case _:
            raise ValueError(f"Unsupported dataset: {SOURCE_DOMAIN}")

### Setup TTA

In [ ]:
tta = methods.MethodContainer(**{
    adaptive_model.model_name: adaptive_model
})

### Continual TTA - Go rounds without reset

In [ ]:
evaluator = DetectionEvaluator(tta.methods(), classes=CLASSES, data_preparation=data_preparation, dtype=DATA_TYPE, device=device, no_grad=False)
evaluator_loader_params = dict(batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_preparation.collate_fn)

In [ ]:
continual_result = []
if continual_scenario:
    for rd, this in tta.go_rounds(end_round=TOTAL_ROUNDS):
        result = visualize_metrics(continual_scenario(**evaluator_loader_params).play(evaluator, index=this))
        continual_result.append(result)
        _, result_path = get_save_path(continual_scenario, rd)
        save_result_json(result, result_path)